In [1]:
!pip install -q yfinance

In [2]:
import os, sys, json, shutil, warnings, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import joblib, sklearn

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix,
                             mean_absolute_error, mean_squared_error, r2_score)
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": 0.3})


PROJECT = Path("nifty50-ml-pipeline")
for sub in ["data", "models", "reports/figures"]:
    (PROJECT / sub).mkdir(parents=True, exist_ok=True)


def savefig(name):
    plt.tight_layout()
    plt.savefig(PROJECT / "reports" / "figures" / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()


print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

pandas 2.2.3 | scikit-learn 1.6.1


## 1. Problem definition

**Context.** The Nifty 50 is India's benchmark equity index (50 large NSE-listed companies). Investors, risk teams and students of finance all watch two things: *where the market goes next* and *how turbulent it will be*.

**Questions we answer with the same dataset**

| | Task | Type | Target (what the model predicts at the close of day *t*) |
|---|------|------|-------------------------------------------------------|
| Q1 | Will the index close **higher tomorrow**? | Binary classification | `y_dir = 1` if `close(t+1) > close(t)` |
| Q2 | How **volatile** will the next week be? | Regression | `y_vol5` = standard deviation of daily log returns over days `t+1 ... t+5` |

**Why two tasks?** Daily direction is close to a coin flip in an efficient market, so we expect only a small edge and treat an honest "barely beats the baseline" as a valid finding. Volatility is far more predictable (it clusters), which gives the project a result with real practical value, for example a risk dashboard.

**Evaluation plan (decided before looking at any results)**
- **Chronological split**: first 80% of days for training, last 20% for testing, with a **5-day gap (purge)** so the 5-day volatility target cannot leak across the boundary. No random shuffling.
- **Walk-forward cross-validation** (`TimeSeriesSplit`) on the training part only, for model selection and tuning.
- **Baselines the models must beat:** majority class and "yesterday's direction repeats" (Q1); persistence of recent volatility and the long-run mean (Q2).
- **Metrics:** accuracy, balanced accuracy, precision, recall, F1, ROC-AUC (Q1); MAE, RMSE, R² (Q2).
- **Success criteria:** Q1, test ROC-AUC clearly above 0.5 and accuracy above the majority baseline (tested with a binomial test). Q2, lower MAE than the best persistence baseline.

**Assumptions and limits.** One row = one trading day. Only index-level price (and volume if usable) data, no news, no macro data. The index itself is not directly tradable, and the backtest ignores slippage and taxes. This is a learning project, not investment advice.

## 2. Data collection

Source: Kaggle dataset *NIFTY 50 Stock Market Data (2000-2023)*, downloaded as a CSV file.
Update `DATA_PATH` to the exact file name, or just run the cell and upload the file when asked.
(If the Kaggle download is a zip with several CSVs, extract it and use the file for the index itself.)

In [ ]:
DATA_PATH = "Nifty 50.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Select the Kaggle CSV file to upload ...")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Put the CSV next to this notebook or fix DATA_PATH ({DATA_PATH!r}).")

raw = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH!r}")
print(f"Rows: {raw.shape[0]:,}   Columns: {raw.shape[1]}")

Select the Kaggle CSV file to upload ...


### 2.1 What are the rows and columns?
The cells below print the real structure of *your* file. Each **row** should be one trading day of the index and each **column** one attribute of that day.

In [ ]:
print("COLUMN NAMES AND DATA TYPES")
display(raw.dtypes.to_frame("dtype").assign(non_null=raw.notna().sum(), missing=raw.isna().sum()))

print("FIRST 5 ROWS")
display(raw.head())
print("LAST 5 ROWS")
display(raw.tail())

In [ ]:
print("SUMMARY STATISTICS (all columns)")
display(raw.describe(include="all").T)

print("Duplicate rows:", int(raw.duplicated().sum()))

**Data dictionary** (typical layout for an index price file; confirm against the output above and edit this table):

| Column | Meaning |
|--------|---------|
| Date | Trading day (one row per day) |
| Open | Index level at the market open |
| High | Highest level during the day |
| Low | Lowest level during the day |
| Close | Index level at the close (the main series we model) |
| Volume | Shares traded (may be missing/zero for an index in some years) |

Any other columns (for example *Adj Close*, *Turnover*, *Change %*) are listed by the cell above and are either derived from these or not needed.

## 3. Data preparation

Steps: standardise names, parse dates and numbers, drop invalid rows, check gaps and outliers, decide whether volume is usable, then engineer features.
The code is written to cope with the common layouts of Kaggle index files (ISO or `dd-mm-yyyy` dates, numbers with commas, `1.2M` volume strings, `Price` instead of `Close`).

In [ ]:
df = raw.copy()
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))


rename = {}
for col in df.columns:
    if col in {"datetime", "timestamp", "trade_date"}:
        rename[col] = "date"
    elif col in {"vol", "shares_traded", "traded_quantity"}:
        rename[col] = "volume"
if "close" not in df.columns:
    for alt in ("price", "close_last", "closing_price", "last"):
        if alt in df.columns:
            rename[alt] = "close"
            break
df = df.rename(columns=rename)


SYMBOL = None
id_cols = [c for c in ("symbol", "ticker", "company", "name") if c in df.columns]
for c in id_cols:
    if df[c].nunique() > 1:
        if SYMBOL is None:
            raise ValueError(f"Column {c!r} has {df[c].nunique()} different values. Set SYMBOL to one of: "
                             f"{df[c].unique()[:10].tolist()} ...")
        df = df[df[c] == SYMBOL]

KEEP = [c for c in ["date", "open", "high", "low", "close", "volume"] if c in df.columns]
assert {"date", "close"} <= set(KEEP), f"Need at least a date and a close column, found {list(df.columns)}"
print("Using columns  :", KEEP)
print("Ignoring columns:", [c for c in df.columns if c not in KEEP])
df = df[KEEP].copy()

In [ ]:
def parse_dates(s):
    s = s.astype(str).str.strip()
    parts = s.str.extract(r"^(\d{1,2})[-/.](\d{1,2})[-/.](\d{4})")
    if parts.notna().all(axis=1).mean() > 0.9:
        first, second = pd.to_numeric(parts[0]), pd.to_numeric(parts[1])
        if (first > 12).any():
            dayfirst = True
        elif (second > 12).any():
            dayfirst = False
        else:
            dayfirst = True
        print(f"Date format looks like {'day-first' if dayfirst else 'month-first'}")
        return pd.to_datetime(s, dayfirst=dayfirst, errors="coerce")
    return pd.to_datetime(s, errors="coerce")


def to_number(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    s = s.astype(str).str.strip().str.replace(",", "", regex=False).str.replace("%", "", regex=False)
    s = s.replace({"": np.nan, "-": np.nan, "nan": np.nan, "None": np.nan})
    mult = (s.str.extract(r"([KMBkmb])$")[0].str.upper()
            .map({"K": 1e3, "M": 1e6, "B": 1e9}).fillna(1.0))
    num = pd.to_numeric(s.str.replace(r"[KMBkmb]$", "", regex=True), errors="coerce")
    return num * mult


n_start = len(df)
df["date"] = parse_dates(df["date"])
for c in ["open", "high", "low", "close", "volume"]:
    if c in df.columns:
        df[c] = to_number(df[c])

bad_dates = int(df["date"].isna().sum())
df = df.dropna(subset=["date", "close"])
dups = int(df["date"].duplicated().sum())
df = df.sort_values("date").drop_duplicates("date", keep="last").set_index("date")


bad_price = df["close"] <= 0
if {"high", "low"} <= set(df.columns):
    bad_price = bad_price | (df["high"] < df["low"])
n_bad = int(bad_price.sum())
df = df[~bad_price]


for c in ("open", "high", "low"):
    if c in df.columns and df[c].isna().any():
        print(f"Filled {int(df[c].isna().sum())} missing {c!r} values with the close")
        df[c] = df[c].fillna(df["close"])


USE_VOLUME = False
if "volume" in df.columns:
    coverage = float((df["volume"].fillna(0) > 0).mean())
    USE_VOLUME = coverage >= 0.8
    print(f"Volume present on {coverage:.0%} of days -> {'used' if USE_VOLUME else 'NOT used'} as a feature")

cleaning_log = pd.Series({
    "rows in raw file": n_start, "unparseable dates / missing close dropped": bad_dates,
    "duplicate dates removed": dups, "invalid price rows removed": n_bad,
    "rows after cleaning": len(df), "first date": df.index.min().date(), "last date": df.index.max().date(),
})
display(cleaning_log.to_frame("value"))

In [ ]:
gap_days = df.index.to_series().diff().dt.days
print("Largest gaps between consecutive trading days (calendar days):")
display(gap_days.sort_values(ascending=False).head(5).to_frame("gap_days"))
print("Rows per year (about 245-250 expected for a full year):")
display(df.groupby(df.index.year).size().to_frame("trading_days").T)


daily_move = df["close"].pct_change()
print("Days with an absolute move above 7%:")
display((daily_move[daily_move.abs() > 0.07] * 100).round(2).to_frame("daily_return_%"))

df.to_csv(PROJECT / "data" / "nifty50_clean.csv")
print("Saved the cleaned data to", PROJECT / "data" / "nifty50_clean.csv")
df.head()

## 4. Exploratory data analysis (EDA)

In [ ]:
n_panels = 2 if USE_VOLUME else 1
fig, axes = plt.subplots(n_panels, 1, sharex=True, figsize=(11, 3.2 * n_panels + 1), squeeze=False)
axes[0, 0].plot(df.index, df["close"], lw=1)
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("Nifty 50 close (log scale)")
if USE_VOLUME:
    axes[1, 0].plot(df.index, df["volume"], lw=0.6, color="tab:gray")
    axes[1, 0].set_title("Volume")
savefig("01_price")

In [ ]:
r = np.log(df["close"]).diff().dropna()
jb = stats.jarque_bera(r)
summary = pd.Series({
    "trading days": len(r), "mean daily return %": r.mean() * 100, "std daily %": r.std() * 100,
    "annualised volatility %": r.std() * np.sqrt(252) * 100, "annualised mean return %": r.mean() * 252 * 100,
    "skewness": r.skew(), "excess kurtosis": r.kurt(), "Jarque-Bera p-value": jb.pvalue,
    "% up days": (r > 0).mean() * 100,
}).round(4)
display(summary.to_frame("value"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(r, bins=120, density=True, alpha=0.7)
xs = np.linspace(r.min(), r.max(), 400)
axes[0].plot(xs, stats.norm.pdf(xs, r.mean(), r.std()), "r--", label="normal with same mean/std")
axes[0].set_title("Daily log returns"); axes[0].legend()
stats.probplot(r, dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q plot (tails off the line = fat tails)")
savefig("02_return_distribution")

In [ ]:
roll_vol = r.rolling(21).std() * np.sqrt(252) * 100
plt.plot(roll_vol.index, roll_vol, lw=0.9)
plt.title("Rolling 21-day annualised volatility (%)")
savefig("03_rolling_volatility")

lags = range(1, 21)
acf_r = [r.autocorr(l) for l in lags]
acf_abs = [r.abs().autocorr(l) for l in lags]
band = 1.96 / np.sqrt(len(r))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, vals, title in zip(axes, [acf_r, acf_abs], ["returns", "absolute returns"]):
    ax.bar(list(lags), vals)
    ax.axhline(band, ls="--", c="r", lw=0.8); ax.axhline(-band, ls="--", c="r", lw=0.8)
    ax.set_title(f"Autocorrelation of {title}"); ax.set_xlabel("lag (trading days)")
savefig("04_autocorrelation")

In [ ]:
tmp = pd.DataFrame({"r": r})
tmp["dow"] = tmp.index.dayofweek
tmp["month"] = tmp.index.month
tmp["year"] = tmp.index.year
day_names = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri"}

dow = tmp.groupby("dow")["r"].agg(mean_pct=lambda x: x.mean() * 100, up_share_pct=lambda x: (x > 0).mean() * 100)
dow.index = [day_names.get(i, str(i)) for i in dow.index]
display(dow.round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
yearly = (np.exp(tmp.groupby("year")["r"].sum()) - 1) * 100
axes[0].bar(yearly.index.astype(str), yearly.values, color=np.where(yearly.values >= 0, "tab:green", "tab:red"))
axes[0].set_title("Calendar-year return (%)"); axes[0].tick_params(axis="x", rotation=90)
axes[1].bar(dow.index, dow["mean_pct"]); axes[1].set_title("Mean daily return by weekday (%)")
month_mean = tmp.groupby("month")["r"].mean() * 100
axes[2].bar(month_mean.index, month_mean.values); axes[2].set_title("Mean daily return by month (%)")
savefig("05_calendar_and_yearly")

In [ ]:
equity = df["close"]
drawdown = equity / equity.cummax() - 1
plt.fill_between(drawdown.index, drawdown * 100, 0, color="tab:red", alpha=0.5)
plt.title(f"Drawdown from the previous peak (%), worst = {drawdown.min():.1%}")
savefig("06_drawdown")

extremes = pd.concat([r.nsmallest(5).rename("worst 5 days"), r.nlargest(5).rename("best 5 days")], axis=1)
display(((np.exp(r.nsmallest(5)) - 1) * 100).round(2).to_frame("worst 5 days (%)"))
display(((np.exp(r.nlargest(5)) - 1) * 100).round(2).to_frame("best 5 days (%)"))

### 4.6 EDA insights (replace the prompts below with your own observations after reading the plots)

Things the plots usually show for equity indices. **Verify each against your own output before keeping it:**
1. *Trend:* long-run upward drift with large drawdowns in stress periods; write down the worst drawdown and its dates.
2. *Fat tails:* the Q-Q plot bends away from the line and the Jarque-Bera test rejects normality, so crashes are more common than a normal model suggests.
3. *Volatility clustering:* autocorrelation of returns is near zero, but that of absolute returns is clearly positive for many lags, which makes the volatility task predictable and the direction task hard.
4. *Calendar effects:* weekday and month averages are small relative to their noise; do not over-interpret them.
5. *Class balance:* the share of up days is slightly above 50%, so "always up" is a non-trivial baseline for Q1.

**Our observations:** _(write 4-6 bullet points here, with numbers from the outputs above)_

## 5. Model building

### 5.1 Feature engineering (leakage-safe)
Every feature at row *t* uses only data up to the close of day *t*; targets look forward. The feature code lives in `features.py`, which is also imported by the deployed app, so training and serving cannot drift apart.

| Group | Features |
|-------|----------|
| Recent returns | `ret_t0 ... ret_t4` (today and the previous 4 daily log returns) |
| Momentum | 5/10/20-day price change |
| Trend | distance of the close from its 5/10/20/50-day moving average |
| Volatility | rolling std of returns over 5/10/20 days |
| Oscillators | RSI(14), MACD and MACD histogram (scaled by price) |
| Candle shape | high-low range, open-to-close move, overnight gap (if open/high/low exist) |
| Volume | volume relative to its 20-day average (only if volume is usable) |
| Calendar | day of week, month |

In [ ]:
%%writefile nifty50-ml-pipeline/features.py
import numpy as np
import pandas as pd


def prepare_ohlcv(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    df.index = pd.to_datetime(df.index)
    if getattr(df.index, "tz", None) is not None:
        df.index = df.index.tz_localize(None)
    df = df[~df.index.duplicated(keep="last")].sort_index()
    return df.dropna(subset=["close"])


def rsi(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    out = 100 - 100 / (1 + rs)
    out[(loss == 0) & (gain > 0)] = 100.0
    return out


def build_features(df, use_volume=True):
    close = df["close"]
    ret = np.log(close).diff()
    f = pd.DataFrame(index=df.index)

    for k in range(5):
        f[f"ret_t{k}"] = ret.shift(k)
    for w in (5, 10, 20):
        f[f"mom_{w}d"] = close.pct_change(w)
    for w in (5, 10, 20, 50):
        f[f"close_vs_sma{w}"] = close / close.rolling(w).mean() - 1
    for w in (5, 10, 20):
        f[f"vol_{w}"] = ret.rolling(w).std()

    f["rsi_14"] = rsi(close, 14)
    macd = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
    signal = macd.ewm(span=9, adjust=False).mean()
    f["macd"] = macd / close
    f["macd_hist"] = (macd - signal) / close

    if {"high", "low"} <= set(df.columns):
        f["hl_range"] = (df["high"] - df["low"]) / close
    if "open" in df.columns:
        f["oc_move"] = close / df["open"] - 1
        f["overnight_gap"] = df["open"] / close.shift(1) - 1
    if use_volume and "volume" in df.columns:
        vol = df["volume"].replace(0, np.nan)
        f["volume_vs_avg20"] = vol / vol.rolling(20).mean() - 1

    f["day_of_week"] = df.index.dayofweek
    f["month"] = df.index.month
    return f.replace([np.inf, -np.inf], np.nan)


def latest_features(df, feature_cols, use_volume=True):
    feats = build_features(prepare_ohlcv(df), use_volume=use_volume)
    missing = [c for c in feature_cols if c not in feats.columns]
    if missing:
        raise ValueError(f"Input data lacks the columns needed for these features: {missing}")
    last = feats[feature_cols].iloc[[-1]]
    if last.isna().any(axis=1).iloc[0]:
        raise ValueError("Not enough clean history to compute every feature (about 60+ trading days needed).")
    return last

In [ ]:
sys.path.insert(0, str(PROJECT))
import features
importlib.reload(features)
from features import build_features

X_all = build_features(df, use_volume=USE_VOLUME)
FEATURES = list(X_all.columns)


close = df["close"]
log_ret = np.log(close).diff()
targets = pd.DataFrame(index=df.index)
targets["next_ret"] = close.shift(-1) / close - 1
targets["y_dir"] = (targets["next_ret"] > 0).astype(float)
targets.loc[targets["next_ret"].isna(), "y_dir"] = np.nan
targets["y_vol5"] = log_ret.rolling(5).std().shift(-5)

data = X_all.join(targets)
model_df = data.dropna()
print(f"{len(FEATURES)} features, {len(model_df):,} labelled rows ({model_df.index.min().date()} to {model_df.index.max().date()})")
print("Features:", FEATURES)

In [ ]:
GAP = 5
split = int(len(model_df) * 0.8)
train = model_df.iloc[:split]
test = model_df.iloc[split + GAP:]

X_train, X_test = train[FEATURES], test[FEATURES]
y_train, y_test = train["y_dir"].astype(int), test["y_dir"].astype(int)

print(f"Train: {len(train):,} days  {train.index.min().date()} -> {train.index.max().date()}   up-days {y_train.mean():.1%}")
print(f"Test : {len(test):,} days  {test.index.min().date()} -> {test.index.max().date()}   up-days {y_test.mean():.1%}")

In [ ]:
corr = train[FEATURES + ["next_ret"]].corr()["next_ret"].drop("next_ret").sort_values()
fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 1.2]})
axes[0].barh(corr.index, corr.values)
axes[0].set_title("Correlation of each feature with next-day return")
cm = train[FEATURES].corr()
im = axes[1].imshow(cm, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(FEATURES))); axes[1].set_xticklabels(FEATURES, rotation=90, fontsize=7)
axes[1].set_yticks(range(len(FEATURES))); axes[1].set_yticklabels(FEATURES, fontsize=7)
axes[1].set_title("Feature-feature correlation"); axes[1].grid(False)
plt.colorbar(im, ax=axes[1], fraction=0.046)
savefig("07_feature_correlations")

### 5.4 Q1: next-day direction (classification)
Baselines first, then three model families tuned with walk-forward cross-validation on the training data only.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

clf_candidates = {
    "Logistic Regression": (
        Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=2000))]),
        {"model__C": [0.01, 0.1, 1.0]}),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
        {"max_depth": [3, 5], "min_samples_leaf": [20, 50]}),
    "Gradient Boosting": (
        HistGradientBoostingClassifier(random_state=SEED),
        {"max_depth": [2, 3], "learning_rate": [0.03, 0.1], "max_iter": [100], "l2_regularization": [1.0]}),
}

best_clf, cv_rows = {}, []
for name, (est, grid) in clf_candidates.items():
    gs = GridSearchCV(est, grid, cv=tscv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
    best_clf[name] = gs.best_estimator_
    i = gs.best_index_
    cv_rows.append({"model": name, "cv_auc_mean": gs.cv_results_["mean_test_score"][i],
                    "cv_auc_std": gs.cv_results_["std_test_score"][i], "best_params": gs.best_params_})

cv_table = pd.DataFrame(cv_rows).set_index("model")
display(cv_table.round(4))
BEST_CLF_NAME = cv_table["cv_auc_mean"].idxmax()
print("Selected for Q1:", BEST_CLF_NAME)

### 5.5 Q2: next-5-day volatility (regression)

In [ ]:
y_tr_v, y_te_v = train["y_vol5"], test["y_vol5"]

reg_candidates = {
    "Ridge": (
        Pipeline([("scale", StandardScaler()), ("model", Ridge())]),
        {"model__alpha": [0.1, 1.0, 10.0, 100.0]}),
    "Random Forest": (
        RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
        {"max_depth": [3, 5], "min_samples_leaf": [20, 50]}),
    "Gradient Boosting": (
        HistGradientBoostingRegressor(random_state=SEED),
        {"max_depth": [2, 3], "learning_rate": [0.03, 0.1], "max_iter": [100], "l2_regularization": [1.0]}),
}

best_reg, reg_cv_rows = {}, []
for name, (est, grid) in reg_candidates.items():
    gs = GridSearchCV(est, grid, cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1).fit(X_train, y_tr_v)
    best_reg[name] = gs.best_estimator_
    i = gs.best_index_
    reg_cv_rows.append({"model": name, "cv_mae_%": -gs.cv_results_["mean_test_score"][i] * 100,
                        "cv_mae_std_%": gs.cv_results_["std_test_score"][i] * 100, "best_params": gs.best_params_})

reg_cv_table = pd.DataFrame(reg_cv_rows).set_index("model")
display(reg_cv_table.round(4))
BEST_REG_NAME = reg_cv_table["cv_mae_%"].idxmin()
print("Selected for Q2:", BEST_REG_NAME)

## 6. Model evaluation
All numbers below come from the **held-out test period**, which no model or hyper-parameter choice has seen.

### 6.1 Q1: direction

In [ ]:
def clf_metrics(y_true, y_pred, y_prob=None):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan,
    }


rows, test_prob, test_pred = {}, {}, {}


majority = int(y_train.mean() >= 0.5)
rows["Baseline: always majority class"] = clf_metrics(y_test, np.full(len(y_test), majority))
rows["Baseline: yesterday repeats"] = clf_metrics(y_test, (X_test["ret_t0"] > 0).astype(int))


for name, model in best_clf.items():
    prob = model.predict_proba(X_test)[:, 1]
    test_prob[name] = prob
    test_pred[name] = (prob >= 0.5).astype(int)
    rows[name] = clf_metrics(y_test, test_pred[name], prob)

clf_results = pd.DataFrame(rows).T
display(clf_results.round(4))
maj_acc = rows["Baseline: always majority class"]["accuracy"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for name, prob in test_prob.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, prob):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8, label="coin flip")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate"); axes[0].legend(fontsize=8)
axes[0].set_title("ROC curves (test period)")

cmx = confusion_matrix(y_test, test_pred[BEST_CLF_NAME])
axes[1].imshow(cmx, cmap="Blues"); axes[1].grid(False)
for (i, j), v in np.ndenumerate(cmx):
    axes[1].text(j, i, int(v), ha="center", va="center", fontsize=13)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["pred down", "pred up"])
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["actual down", "actual up"])
axes[1].set_title(f"Confusion matrix: {BEST_CLF_NAME}")
savefig("08_direction_roc_confusion")


correct = int((test_pred[BEST_CLF_NAME] == y_test.values).sum())
bt = stats.binomtest(correct, n=len(y_test), p=maj_acc, alternative="greater")
print(f"{BEST_CLF_NAME}: {correct}/{len(y_test)} correct = {correct / len(y_test):.3f} vs majority baseline {maj_acc:.3f}")
print(f"One-sided binomial test p-value = {bt.pvalue:.3f}  ->  "
      f"{'significant at 5%' if bt.pvalue < 0.05 else 'NOT significantly better than the baseline'}")

In [ ]:
pi = permutation_importance(best_clf[BEST_CLF_NAME], X_test, y_test, scoring="roc_auc",
                            n_repeats=10, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
err = pd.Series(pi.importances_std, index=FEATURES)[imp.index]
imp.tail(12).plot.barh(xerr=err.tail(12), figsize=(8, 5))
plt.title(f"Permutation importance (drop in test AUC), {BEST_CLF_NAME}")
savefig("09_permutation_importance")

In [ ]:
ea = test[["next_ret", "vol_20"]].copy()
ea["correct"] = (test_pred[BEST_CLF_NAME] == y_test.values).astype(int)

by_year = ea.groupby(ea.index.year)["correct"].agg(accuracy="mean", days="size")
q1, q2 = train["vol_20"].quantile([1 / 3, 2 / 3])
ea["vol_regime"] = pd.cut(ea["vol_20"], [-np.inf, q1, q2, np.inf], labels=["calm", "normal", "turbulent"])
by_regime = ea.groupby("vol_regime", observed=True)["correct"].agg(accuracy="mean", days="size")
display(by_year.round(3).T)
display(by_regime.round(3).T)

In [ ]:
def backtest(next_ret, position, cost=0.0005):
    trades = position.diff().abs().fillna(position.abs())
    return position * next_ret - trades * cost


def perf(r):
    eq = (1 + r).cumprod()
    return {"total return %": (eq.iloc[-1] - 1) * 100,
            "annualised return %": ((eq.iloc[-1]) ** (252 / len(r)) - 1) * 100,
            "annualised volatility %": r.std() * np.sqrt(252) * 100,
            "Sharpe (rf = 0)": r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan,
            "max drawdown %": (eq / eq.cummax() - 1).min() * 100}


nr = test["next_ret"]
position = pd.Series(test_pred[BEST_CLF_NAME], index=test.index).astype(float)
strat = backtest(nr, position)
bh = nr

bt_table = pd.DataFrame({"Model strategy (0.05% cost/trade)": perf(strat), "Buy & hold": perf(bh)}).round(2)
display(bt_table)

(1 + strat).cumprod().plot(label="Model strategy")
(1 + bh).cumprod().plot(label="Buy & hold")
plt.title("Growth of 1 rupee on the test period"); plt.legend()
savefig("10_backtest")

### 6.2 Q2: volatility

In [ ]:
def reg_metrics(y_true, y_pred):
    return {"MAE %": mean_absolute_error(y_true, y_pred) * 100,
            "RMSE %": np.sqrt(mean_squared_error(y_true, y_pred)) * 100,
            "R2": r2_score(y_true, y_pred)}


reg_rows, reg_pred = {}, {}
reg_rows["Baseline: long-run mean"] = reg_metrics(y_te_v, np.full(len(y_te_v), y_tr_v.mean()))
reg_rows["Baseline: last 5-day vol repeats"] = reg_metrics(y_te_v, X_test["vol_5"])
reg_rows["Baseline: last 20-day vol repeats"] = reg_metrics(y_te_v, X_test["vol_20"])
for name, model in best_reg.items():
    reg_pred[name] = model.predict(X_test)
    reg_rows[name] = reg_metrics(y_te_v, reg_pred[name])

reg_results = pd.DataFrame(reg_rows).T
display(reg_results.round(4))

best_base_mae = reg_results.loc[[i for i in reg_results.index if i.startswith("Baseline")], "MAE %"].min()
print(f"Selected model {BEST_REG_NAME}: MAE {reg_results.loc[BEST_REG_NAME, 'MAE %']:.4f}% "
      f"vs best baseline {best_base_mae:.4f}%")

plt.plot(y_te_v.index, y_te_v * 100, lw=0.9, label="actual next-5-day volatility")
plt.plot(y_te_v.index, reg_pred[BEST_REG_NAME] * 100, lw=0.9, label=f"{BEST_REG_NAME} forecast")
plt.title("Daily volatility, % (test period)"); plt.legend()
savefig("11_volatility_forecast")

### 6.3 Conclusions (fill in using your own numbers)
- **Q1 (direction):** _Did the selected model beat both baselines on accuracy and ROC-AUC? Was the binomial test significant? How large is the edge and is it stable across years and volatility regimes?_
- **Q2 (volatility):** _By how much did the best model reduce MAE relative to the best persistence baseline?_
- **Backtest:** _Did the strategy beat buy & hold after costs? Compare drawdown and Sharpe, not just return._
- **Most important features:** _from the permutation-importance plot._
- **Limitations:** single index, no macro or news data, one train/test split, transaction costs simplified, survivorship and regime changes (2008-style crises or 2020 are rare events).
- **Next steps:** add macro features (rates, crude oil, USD/INR, US indices), sentiment from news, probability calibration, and walk-forward re-training.

## 7. Deployment

What "deployment" means for this course project:
1. Re-fit the selected models on **all** labelled data (the test period is no longer needed for honest scoring) and save them with `joblib`.
2. Wrap prediction in a function that works on raw OHLC(V) data, reusing the same `features.py`.
3. Ship a **Streamlit app** (`app.py`) that fetches fresh Nifty data (or accepts a CSV) and shows tomorrow's probability and the volatility forecast.
4. Make the whole thing reproducible: `requirements.txt`, `README.md`, saved metrics and figures.

In [ ]:
final_clf = clone(best_clf[BEST_CLF_NAME]).fit(model_df[FEATURES], model_df["y_dir"].astype(int))
final_vol = clone(best_reg[BEST_REG_NAME]).fit(model_df[FEATURES], model_df["y_vol5"])

joblib.dump(final_clf, PROJECT / "models" / "direction_model.joblib")
joblib.dump(final_vol, PROJECT / "models" / "volatility_model.joblib")

metadata = {
    "created": pd.Timestamp.now().isoformat(timespec="seconds"),
    "data_file": os.path.basename(DATA_PATH),
    "trained_on": {"start": str(model_df.index.min().date()), "end": str(model_df.index.max().date()),
                   "rows": int(len(model_df))},
    "features": FEATURES,
    "use_volume": bool(USE_VOLUME),
    "direction_model": BEST_CLF_NAME,
    "volatility_model": BEST_REG_NAME,
    "test_period": {"start": str(test.index.min().date()), "end": str(test.index.max().date()), "rows": int(len(test))},
    "test_metrics": {
        "direction": {k: float(v) for k, v in clf_results.loc[BEST_CLF_NAME].items()},
        "direction_majority_baseline_accuracy": float(maj_acc),
        "direction_binomial_p_value": float(bt.pvalue),
        "volatility": {k: float(v) for k, v in reg_results.loc[BEST_REG_NAME].items()},
        "volatility_best_baseline_mae_pct": float(best_base_mae),
    },
    "sklearn_version": sklearn.__version__,
}
(PROJECT / "models" / "metadata.json").write_text(json.dumps(metadata, indent=2))
(PROJECT / "reports" / "metrics.json").write_text(json.dumps(metadata["test_metrics"], indent=2))
print("Saved:", sorted(p.name for p in (PROJECT / "models").iterdir()))

In [ ]:
from features import latest_features

def predict_next_day(ohlcv):
    meta = json.loads((PROJECT / "models" / "metadata.json").read_text())
    clf = joblib.load(PROJECT / "models" / "direction_model.joblib")
    vol = joblib.load(PROJECT / "models" / "volatility_model.joblib")
    x = latest_features(ohlcv, meta["features"], use_volume=meta["use_volume"])
    p_up = float(clf.predict_proba(x)[0, 1])
    vol_daily = float(vol.predict(x)[0])
    return {"as_of": str(x.index[0].date()), "prob_up_next_day": round(p_up, 4),
            "signal": "UP" if p_up >= 0.5 else "DOWN",
            "expected_daily_vol_next_5d_%": round(vol_daily * 100, 3),
            "expected_annualised_vol_%": round(float(vol_daily * np.sqrt(252) * 100), 2)}


print(predict_next_day(df))

In [ ]:
try:
    import yfinance as yf
    live = yf.download("^NSEI", period="2y", auto_adjust=False, progress=False)
    print(predict_next_day(live))
except Exception as e:
    print("Live-data demo skipped:", e)

### 7.4 Repository files
The next cells write the Streamlit app and the project documentation into the repo folder.

In [ ]:
%%writefile nifty50-ml-pipeline/app.py
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import streamlit as st

from features import latest_features, prepare_ohlcv

ROOT = Path(__file__).parent
MODELS = ROOT / "models"

st.set_page_config(page_title="Nifty 50 forecast", layout="wide")
st.title("Nifty 50: next-day direction and short-term volatility")
st.caption("Educational project built for the IDSE course. Not investment advice.")


@st.cache_resource
def load_artifacts():
    meta = json.loads((MODELS / "metadata.json").read_text())
    clf = joblib.load(MODELS / "direction_model.joblib")
    vol = joblib.load(MODELS / "volatility_model.joblib")
    return meta, clf, vol


@st.cache_data(ttl=3600)
def fetch_live(ticker, period):
    import yfinance as yf
    return yf.download(ticker, period=period, auto_adjust=False, progress=False)


def read_uploaded(file):
    df = pd.read_csv(file)
    date_col = next((c for c in df.columns if c.strip().lower() in ("date", "datetime", "timestamp")), None)
    if date_col is None:
        raise ValueError("The CSV needs a Date column.")
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")
    return df.dropna(subset=[date_col]).set_index(date_col)


meta, clf, vol_model = load_artifacts()
if meta.get("sklearn_version") != sklearn.__version__:
    st.warning(f"Models were trained with scikit-learn {meta.get('sklearn_version')}, "
               f"this environment has {sklearn.__version__}. Results may differ or fail to load.")

with st.sidebar:
    st.header("Data source")
    source = st.radio("Where should the latest prices come from?", ["Live (Yahoo Finance)", "Upload CSV"])
    ticker = st.text_input("Ticker", "^NSEI") if source.startswith("Live") else None
    upload = st.file_uploader("CSV with Date, Open, High, Low, Close (Volume optional)") if source == "Upload CSV" else None
    st.markdown("---")
    st.write("**Models**")
    st.write(f"Direction: {meta['direction_model']}")
    st.write(f"Volatility: {meta['volatility_model']}")
    st.write(f"Trained on {meta['trained_on']['start']} to {meta['trained_on']['end']}")

try:
    if source.startswith("Live"):
        raw = fetch_live(ticker, "3y")
    elif upload is not None:
        raw = read_uploaded(upload)
    else:
        st.info("Upload a CSV in the sidebar to continue.")
        st.stop()
    prices = prepare_ohlcv(raw)
    x = latest_features(prices, meta["features"], use_volume=meta["use_volume"])
except Exception as exc:
    st.error(f"Could not prepare the data: {exc}")
    st.stop()

p_up = float(clf.predict_proba(x)[0, 1])
vol_daily = float(vol_model.predict(x)[0])

c1, c2, c3, c4 = st.columns(4)
c1.metric("Latest date", str(x.index[0].date()))
c2.metric("Latest close", f"{prices['close'].iloc[-1]:,.2f}")
c3.metric("Probability up tomorrow", f"{p_up:.1%}", "UP" if p_up >= 0.5 else "DOWN", delta_color="off")
c4.metric("Expected volatility (annualised)", f"{vol_daily * np.sqrt(252):.1%}",
          f"{vol_daily:.2%} per day", delta_color="off")

st.subheader("Last 250 trading days")
st.line_chart(prices["close"].tail(250))

with st.expander("Features used for this prediction"):
    st.dataframe(x.T.rename(columns={x.index[0]: "value"}))

tm = meta["test_metrics"]
with st.expander("How good is the model? (held-out test period)"):
    st.write(f"Direction: accuracy {tm['direction']['accuracy']:.1%} vs majority baseline "
             f"{tm['direction_majority_baseline_accuracy']:.1%}, ROC-AUC {tm['direction']['roc_auc']:.3f}.")
    st.write(f"Volatility: MAE {tm['volatility']['MAE %']:.3f}% per day vs best naive baseline "
             f"{tm['volatility_best_baseline_mae_pct']:.3f}%.")
    st.write("Daily direction is intrinsically hard to predict; treat the probability as weak evidence, "
             "and the volatility forecast as the more reliable output.")

In [ ]:
%%writefile nifty50-ml-pipeline/requirements.txt
numpy
pandas
scipy
scikit-learn
matplotlib
joblib
streamlit
yfinance

In [ ]:
%%writefile nifty50-ml-pipeline/.gitignore
__pycache__/
.ipynb_checkpoints/
.DS_Store
*.zip
data/*.csv

In [ ]:
%%writefile nifty50-ml-pipeline/data/README.md
# Data

The raw dataset is not stored in this repository.

1. Download **NIFTY 50 Stock Market Data (2000-2023)** from Kaggle:
   https://www.kaggle.com/datasets/rockyjoseph/nifty-50-stock-market-data-2000-2023
2. Check the licence shown on the Kaggle page before redistributing the file.
3. Place the CSV in this folder (or upload it when the notebook asks for it).

`nifty50_clean.csv` is produced by the notebook (cleaned, one row per trading day).

In [ ]:
%%writefile nifty50-ml-pipeline/README.md
# Nifty 50: next-day direction and volatility forecasting

End-to-end data science pipeline for the IDSE course: problem definition, data collection, preparation,
EDA, model building, evaluation and deployment (Streamlit app).

**Team:** _add names and roll numbers_

## Problem
Using daily Nifty 50 index data (2000-2023) we answer two questions at the close of each trading day:
1. **Direction (classification):** will the index close higher tomorrow?
2. **Volatility (regression):** what will the standard deviation of daily returns be over the next 5 trading days?

## Data
Kaggle: *NIFTY 50 Stock Market Data (2000-2023)* by Rocky Joseph
(https://www.kaggle.com/datasets/rockyjoseph/nifty-50-stock-market-data-2000-2023). See `data/README.md`.

## Method
- Cleaning and validation (date parsing, duplicates, impossible prices, gaps, volume usability)
- Leakage-safe features (returns, momentum, moving-average distance, volatility, RSI, MACD, candle shape, calendar) in `features.py`
- Chronological 80/20 split with a 5-day purge gap; walk-forward `TimeSeriesSplit` cross-validation on the training part
- Baselines (majority class, "yesterday repeats", volatility persistence) versus Logistic/Ridge, Random Forest and Gradient Boosting
- Evaluation: accuracy, balanced accuracy, precision, recall, F1, ROC-AUC, binomial significance test, permutation importance,
  error analysis by year and volatility regime, cost-aware backtest

## Repository layout
```
nifty50-ml-pipeline/
  notebooks/        Colab notebook (full pipeline)
  features.py       feature engineering shared by notebook and app
  app.py            Streamlit dashboard
  models/           saved models + metadata.json
  reports/          metrics.json and figures
  data/             data notes (CSV not committed)
  requirements.txt
```

## Reproduce
```bash
pip install -r requirements.txt
# 1) run notebooks/nifty50_pipeline.ipynb (Colab or Jupyter) to regenerate models/ and reports/
# 2) launch the app
streamlit run app.py
```

## Limitations
Index-level data only; daily direction is close to unpredictable, so the directional edge is small and
should not be used for trading. The backtest ignores slippage, taxes and market impact.

In [ ]:
tm = metadata["test_metrics"]
results_md = f'''
## Results (held-out test period {metadata["test_period"]["start"]} to {metadata["test_period"]["end"]})

| Task | Model | Key metrics | Baseline to beat |
|------|-------|-------------|------------------|
| Direction | {BEST_CLF_NAME} | accuracy {tm["direction"]["accuracy"]:.3f}, ROC-AUC {tm["direction"]["roc_auc"]:.3f}, F1 {tm["direction"]["f1"]:.3f} | majority-class accuracy {tm["direction_majority_baseline_accuracy"]:.3f} (binomial p = {tm["direction_binomial_p_value"]:.3f}) |
| Volatility (5-day) | {BEST_REG_NAME} | MAE {tm["volatility"]["MAE %"]:.3f}% per day, R2 {tm["volatility"]["R2"]:.3f} | best persistence MAE {tm["volatility_best_baseline_mae_pct"]:.3f}% |

Figures are in `reports/figures/`.
'''
with open(PROJECT / "README.md", "a") as fh:
    fh.write(results_md)


(PROJECT / "requirements-lock.txt").write_text(
    "\n".join([f"numpy=={np.__version__}", f"pandas=={pd.__version__}", f"scikit-learn=={sklearn.__version__}",
               f"scipy=={__import__('scipy').__version__}", f"joblib=={joblib.__version__}"]) + "\n")
print(results_md)

### 7.5 Run the app (optional)
Locally: `pip install -r requirements.txt` then `streamlit run app.py` inside the downloaded folder.

To try it from Colab, you can expose it with a tunnel (slower and less reliable than running locally):
```
!pip install -q streamlit
!streamlit run nifty50-ml-pipeline/app.py &>/dev/null &
!npx localtunnel --port 8501
```
Open the printed URL; the localtunnel password is the Colab VM's public IP (`!curl ipv4.icanhazip.com`).

## 8. Export and publish to GitHub
1. Run the next cell: it zips `nifty50-ml-pipeline/` and downloads it.
2. Save this notebook too: `File -> Download -> Download .ipynb`, then put it in the repo's `notebooks/` folder. (Alternative: `File -> Save a copy in GitHub` pushes the notebook straight to a repo you choose.)
3. Create an empty repository on GitHub, then from the unzipped folder:
```bash
git init
git add .
git commit -m "Nifty 50 pipeline: EDA, models, Streamlit app"
git branch -M main
git remote add origin https://github.com/<your-username>/<repo-name>.git
git push -u origin main
```
4. Check that the README renders, the figures are in `reports/figures/`, and `models/` is committed (the app needs it).

In [ ]:
shutil.make_archive("nifty50-ml-pipeline", "zip", root_dir=".", base_dir=str(PROJECT))
print("Created nifty50-ml-pipeline.zip")
try:
    from google.colab import files
    files.download("nifty50-ml-pipeline.zip")
except ImportError:
    print("Not running in Colab: find the zip next to the notebook.")